<!-- Cell 1 -->
# AquaSelect Cross-Dataset Validation -- Notebook SA-1
# Sea Animals Image Dataset: ConvNeXt-Tiny Training (3 seeds)

**Purpose**: Fine-tune ConvNeXt-Tiny on Sea Animals dataset, save checkpoints + val/test logits.  
**Output**: 3 checkpoints + `sea_animals_logits.pth` for Notebook SA-2.  
**Hardware**: Kaggle T4 GPU  
**Cross-dataset validation**: Same framework as AQUA20, different dataset.

In [ ]:
# Cell 2
!pip install -q timm

import os
import copy
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict
from pathlib import Path

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader, Subset

import timm
from torchvision import transforms
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Cell 3
# Configuration 
CFG = {
    "model_name": "convnext_tiny",
    "image_size": 224,
    "batch_size": 32,
    "num_workers": 2,
    "seeds": [9, 42, 2003],
    # Phase 1: backbone frozen
    "phase1_epochs": 5,
    "phase1_lr": 1e-3,
    # Phase 2: full fine-tuning
    "phase2_epochs": 45,
    "phase2_lr": 1e-4,
    "weight_decay": 1e-4,
    "patience": 10,
    # Splits
    "test_ratio": 0.20,       # 80/20 train+val / test
    "val_ratio": 0.15,        # 85/15 train / val (from train+val)
    "test_split_seed": 0,     # fixed seed for train+val vs test
    "val_split_seed": 42,     # fixed seed for train vs val
    # Paths
    "data_path": "/kaggle/input/datasets/vencerlanz09/sea-animals-image-dataste",
    "output_dir": "/kaggle/working",
}

print("Configuration:")
for k, v in CFG.items():
    print(f"  {k}: {v}")

In [ ]:
# Cell 4
# Explore dataset structure 

data_root = Path(CFG["data_path"])
print(f"Dataset root: {data_root}")
print(f"Contents: {os.listdir(data_root)}")

# Find the actual image directory
candidate_dirs = [data_root]
for item in data_root.iterdir():
    if item.is_dir():
        candidate_dirs.append(item)
        for sub in item.iterdir():
            if sub.is_dir():
                candidate_dirs.append(sub)

# Find the directory that contains class subfolders with images
img_dir = None
for d in candidate_dirs:
    subdirs = [x for x in d.iterdir() if x.is_dir()]
    if len(subdirs) >= 10: 
        sample_subdir = subdirs[0]
        image_files = list(sample_subdir.glob("*.jpg")) + list(sample_subdir.glob("*.png")) + \
                      list(sample_subdir.glob("*.jpeg")) + list(sample_subdir.glob("*.JPG"))
        if len(image_files) > 0:
            img_dir = d
            break

if img_dir is None:
    print("\nCould not auto-detect image directory. Listing all:")
    for p in sorted(data_root.rglob("*"))[:50]:
        print(f"  {p.relative_to(data_root)}")
    raise ValueError("Set img_dir manually based on the listing above")

print(f"\nImage directory: {img_dir}")

# Get class names and counts
class_names = sorted([d.name for d in img_dir.iterdir() if d.is_dir()])
num_classes = len(class_names)
print(f"Number of classes: {num_classes}")

# Count images per class
all_image_paths = []
all_labels = []
class_counts = {}

for class_idx, class_name in enumerate(class_names):
    class_dir = img_dir / class_name
    images = list(class_dir.glob("*.jpg")) + list(class_dir.glob("*.png")) + \
             list(class_dir.glob("*.jpeg")) + list(class_dir.glob("*.JPG")) + \
             list(class_dir.glob("*.JPEG"))
    class_counts[class_name] = len(images)
    for img_path in images:
        all_image_paths.append(str(img_path))
        all_labels.append(class_idx)

print(f"\nTotal images: {len(all_image_paths)}")
print(f"\nPer-class counts:")
for name in class_names:
    print(f"  {name}: {class_counts[name]}")

# Update CFG
CFG["num_classes"] = num_classes
print(f"\nnum_classes set to: {num_classes}")

In [ ]:
# Cell 5
# Create stratified train/val/test splits 

all_indices = list(range(len(all_image_paths)))

# Step 1: train+val vs test
trainval_indices, test_indices = train_test_split(
    all_indices,
    test_size=CFG["test_ratio"],
    random_state=CFG["test_split_seed"],
    stratify=all_labels,
)

# Step 2: train vs val
trainval_labels = [all_labels[i] for i in trainval_indices]
train_rel_indices, val_rel_indices = train_test_split(
    range(len(trainval_indices)),
    test_size=CFG["val_ratio"],
    random_state=CFG["val_split_seed"],
    stratify=trainval_labels,
)

train_indices = [trainval_indices[i] for i in train_rel_indices]
val_indices = [trainval_indices[i] for i in val_rel_indices]

print(f"Train: {len(train_indices)} images")
print(f"Val:   {len(val_indices)} images")
print(f"Test:  {len(test_indices)} images")
print(f"Total: {len(train_indices) + len(val_indices) + len(test_indices)}")

# Verify no overlap
assert len(set(train_indices) & set(val_indices)) == 0
assert len(set(train_indices) & set(test_indices)) == 0
assert len(set(val_indices) & set(test_indices)) == 0
print("No overlap between splits. ✓")

In [ ]:
# Cell 6
# Dataset class, transforms, dataloaders 

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(CFG["image_size"], scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(CFG["image_size"]),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])


class SeaAnimalsDataset(Dataset):
    def __init__(self, image_paths, labels, indices, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.indices = indices
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        real_idx = self.indices[idx]
        image = Image.open(self.image_paths[real_idx]).convert("RGB")
        label = self.labels[real_idx]
        if self.transform:
            image = self.transform(image)
        return image, label


train_dataset = SeaAnimalsDataset(all_image_paths, all_labels, train_indices, train_transform)
val_dataset = SeaAnimalsDataset(all_image_paths, all_labels, val_indices, eval_transform)
test_dataset = SeaAnimalsDataset(all_image_paths, all_labels, test_indices, eval_transform)

train_loader = DataLoader(train_dataset, batch_size=CFG["batch_size"], shuffle=True,
                          num_workers=CFG["num_workers"], pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=CFG["batch_size"], shuffle=False,
                        num_workers=CFG["num_workers"], pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=CFG["batch_size"], shuffle=False,
                         num_workers=CFG["num_workers"], pin_memory=True)

print(f"Train loader: {len(train_loader)} batches x {CFG['batch_size']}")
print(f"Val loader:   {len(val_loader)} batches x {CFG['batch_size']}")
print(f"Test loader:  {len(test_loader)} batches x {CFG['batch_size']}")

#  check
images, labels = next(iter(train_loader))
print(f"\nBatch shape: {images.shape}, Labels: {labels.shape}, Range: {labels.min()}-{labels.max()}")

In [ ]:
# Cell 7
# Model factory and utilities 

class BackboneClassifier(nn.Module):
    def __init__(self, backbone, classifier):
        super().__init__()
        self.backbone = backbone
        self.classifier = classifier
    def forward(self, x):
        features = self.backbone(x)
        logits = self.classifier(features)
        return logits, features


def create_model(model_name="convnext_tiny", num_classes=19):
    backbone = timm.create_model(model_name, pretrained=True, num_classes=0)
    with torch.no_grad():
        feature_dim = backbone(torch.randn(1, 3, 224, 224)).shape[1]
    classifier = nn.Linear(feature_dim, num_classes)
    model = BackboneClassifier(backbone, classifier)
    return model, feature_dim


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def compute_topk_accuracy(logits, labels, k=1):
    topk_preds = logits.topk(k, dim=1).indices
    correct = topk_preds.eq(labels.unsqueeze(1)).any(dim=1).float()
    return correct.mean().item() * 100


def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0
    all_logits, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits, _ = model(images)
            loss = criterion(logits, labels)
            total_loss += loss.item() * labels.size(0)
            all_logits.append(logits.cpu())
            all_labels.append(labels.cpu())
    all_logits = torch.cat(all_logits)
    all_labels = torch.cat(all_labels)
    avg_loss = total_loss / len(all_labels)
    top1 = compute_topk_accuracy(all_logits, all_labels, k=1)
    top3 = compute_topk_accuracy(all_logits, all_labels, k=3)
    macro_f1 = f1_score(all_labels.numpy(), all_logits.argmax(dim=1).numpy(), average="macro") * 100
    return avg_loss, top1, top3, macro_f1, all_logits, all_labels


#  test
_model, _fdim = create_model(CFG["model_name"], CFG["num_classes"])
print(f"Model: {CFG['model_name']}, Feature dim: {_fdim}, "
      f"Params: {sum(p.numel() for p in _model.parameters())/1e6:.1f}M")
del _model

In [ ]:
# Cell 8
# Training function: two-phase transfer learning  

def train_one_seed(seed, train_loader, val_loader, cfg):
    total_epochs = cfg["phase1_epochs"] + cfg["phase2_epochs"]
    
    print(f"\n{'='*70}")
    print(f"  SEED {seed} | Total epochs: {total_epochs} | Batch size: {cfg['batch_size']}")
    print(f"{'='*70}")
    
    set_seed(seed)
    model, feature_dim = create_model(cfg["model_name"], cfg["num_classes"])
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    
    history = {"epoch": [], "train_loss": [], "val_loss": [], "val_top1": [], "val_f1": []}
    global_epoch = 0
    seed_start_time = time.time()
    
    # Phase 1: Backbone frozen 
    print(f"\n--- Phase 1: Classifier warmup (backbone frozen) ---")
    for param in model.backbone.parameters():
        param.requires_grad = False
    optimizer = AdamW(model.classifier.parameters(), lr=cfg["phase1_lr"], weight_decay=cfg["weight_decay"])
    
    for epoch in range(1, cfg["phase1_epochs"] + 1):
        global_epoch += 1
        epoch_start = time.time()
        model.train()
        running_loss, correct, total = 0, 0, 0
        num_batches = len(train_loader)
        
        for batch_idx, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)
            logits, _ = model(images)
            loss = criterion(logits, labels)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            running_loss += loss.item() * labels.size(0)
            correct += (logits.argmax(1) == labels).sum().item()
            total += labels.size(0)
            if (batch_idx + 1) % max(1, num_batches // 5) == 0 or (batch_idx + 1) == num_batches:
                print(f"    P1 Epoch {global_epoch}/{total_epochs} | "
                      f"Batch {batch_idx+1}/{num_batches} ({(batch_idx+1)/num_batches*100:.0f}%) | "
                      f"Loss: {running_loss/total:.4f} | Acc: {correct/total*100:.1f}%", end="\r")
        
        train_loss = running_loss / total
        val_loss, val_top1, _, val_f1, _, _ = evaluate(model, val_loader, criterion)
        epoch_time = time.time() - epoch_start
        elapsed = time.time() - seed_start_time
        
        history["epoch"].append(global_epoch)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_top1"].append(val_top1)
        history["val_f1"].append(val_f1)
        
        print(f"\n  [{global_epoch/total_epochs*100:5.1f}%] P1 Epoch {global_epoch}/{total_epochs} "
              f"({epoch_time:.0f}s, total {elapsed/60:.1f}min) | "
              f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} Top1: {val_top1:.1f}% F1: {val_f1:.1f}%")
    
    # Phase 2: Full fine-tuning 
    print(f"\n--- Phase 2: Full fine-tuning ---")
    for param in model.backbone.parameters():
        param.requires_grad = True
    optimizer = AdamW(model.parameters(), lr=cfg["phase2_lr"], weight_decay=cfg["weight_decay"])
    scheduler = CosineAnnealingLR(optimizer, T_max=cfg["phase2_epochs"], eta_min=1e-6)
    
    best_val_loss = float("inf")
    patience_counter = 0
    best_state = None
    best_epoch = 0
    
    for epoch in range(1, cfg["phase2_epochs"] + 1):
        global_epoch = cfg["phase1_epochs"] + epoch
        epoch_start = time.time()
        model.train()
        running_loss, correct, total = 0, 0, 0
        num_batches = len(train_loader)
        
        for batch_idx, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)
            logits, _ = model(images)
            loss = criterion(logits, labels)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            running_loss += loss.item() * labels.size(0)
            correct += (logits.argmax(1) == labels).sum().item()
            total += labels.size(0)
            if (batch_idx + 1) % max(1, num_batches // 5) == 0 or (batch_idx + 1) == num_batches:
                print(f"    P2 Epoch {global_epoch}/{total_epochs} | "
                      f"Batch {batch_idx+1}/{num_batches} ({(batch_idx+1)/num_batches*100:.0f}%) | "
                      f"Loss: {running_loss/total:.4f} | Acc: {correct/total*100:.1f}%", end="\r")
        
        scheduler.step()
        train_loss = running_loss / total
        val_loss, val_top1, _, val_f1, _, _ = evaluate(model, val_loader, criterion)
        epoch_time = time.time() - epoch_start
        elapsed = time.time() - seed_start_time
        avg_epoch_time = elapsed / global_epoch
        eta_min = avg_epoch_time * (cfg["phase2_epochs"] - epoch) / 60
        
        history["epoch"].append(global_epoch)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_top1"].append(val_top1)
        history["val_f1"].append(val_f1)
        
        improved = ""
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = global_epoch
            improved = " ** BEST **"
        else:
            patience_counter += 1
        
        current_lr = scheduler.get_last_lr()[0]
        print(f"\n  [{global_epoch/total_epochs*100:5.1f}%] P2 Epoch {global_epoch}/{total_epochs} "
              f"({epoch_time:.0f}s, total {elapsed/60:.1f}min, ETA {eta_min:.1f}min) | "
              f"LR: {current_lr:.2e} | Train Loss: {train_loss:.4f} | "
              f"Val Loss: {val_loss:.4f} Top1: {val_top1:.1f}% F1: {val_f1:.1f}% | "
              f"Pat: {patience_counter}/{cfg['patience']}{improved}")
        
        if patience_counter >= cfg["patience"]:
            print(f"\n  ** Early stopping at epoch {global_epoch} **")
            break
    
    total_time = (time.time() - seed_start_time) / 60
    print(f"\n  SEED {seed} COMPLETE | Best epoch: {best_epoch} | "
          f"Best val loss: {best_val_loss:.4f} | Time: {total_time:.1f} min")
    print(f"{'='*70}")
    return best_state, history, feature_dim


print("Training function defined.")

In [ ]:
# Cell 9
# Train across all seeds 

all_results = {}
total_start = time.time()

for seed in CFG["seeds"]:
    seed_start = time.time()
    best_state, history, feature_dim = train_one_seed(seed, train_loader, val_loader, CFG)
    elapsed = (time.time() - seed_start) / 60
    all_results[seed] = {
        "best_state": best_state,
        "history": history,
        "feature_dim": feature_dim,
    }
    print(f"\nSeed {seed} completed in {elapsed:.1f} min")

total_elapsed = (time.time() - total_start) / 60
print(f"\n{'='*70}")
print(f"All seeds completed in {total_elapsed:.1f} min")

In [ ]:
# Cell 10
# Clean evaluation: reload best checkpoints, extract val + test logits 

criterion = nn.CrossEntropyLoss()
seed_metrics = []
saved_outputs = {}

for seed in CFG["seeds"]:
    print(f"\n--- Clean eval for seed {seed} ---")
    
    model, feature_dim = create_model(CFG["model_name"], CFG["num_classes"])
    model.load_state_dict(all_results[seed]["best_state"])
    model = model.to(device)
    model.eval()
    
    val_loss, val_top1, val_top3, val_f1, val_logits, val_labels = evaluate(model, val_loader, criterion)
    print(f"  Val  | Loss: {val_loss:.4f} | Top1: {val_top1:.2f}% | Top3: {val_top3:.2f}% | F1: {val_f1:.2f}%")
    
    test_loss, test_top1, test_top3, test_f1, test_logits, test_labels = evaluate(model, test_loader, criterion)
    print(f"  Test | Loss: {test_loss:.4f} | Top1: {test_top1:.2f}% | Top3: {test_top3:.2f}% | F1: {test_f1:.2f}%")
    
    seed_metrics.append({
        "seed": seed,
        "val_top1": val_top1, "val_top3": val_top3, "val_f1": val_f1,
        "test_top1": test_top1, "test_top3": test_top3, "test_f1": test_f1,
    })
    
    saved_outputs[seed] = {
        "val_logits": val_logits, "val_labels": val_labels,
        "test_logits": test_logits, "test_labels": test_labels,
    }
    
    del model
    torch.cuda.empty_cache()

print("\nClean evaluation complete for all seeds.")

In [ ]:
# Cell 11
# Summary table 

df_metrics = pd.DataFrame(seed_metrics)

print("Per-seed test results:")
print(df_metrics.to_string(index=False))

print(f"\n{'='*60}")
print("ConvNeXt-Tiny on Sea Animals -- Mean +/- Std across 3 seeds:")
print(f"  Top-1 Acc: {df_metrics['test_top1'].mean():.2f} +/- {df_metrics['test_top1'].std():.2f}%")
print(f"  Top-3 Acc: {df_metrics['test_top3'].mean():.2f} +/- {df_metrics['test_top3'].std():.2f}%")
print(f"  Macro F1:  {df_metrics['test_f1'].mean():.2f} +/- {df_metrics['test_f1'].std():.2f}%")

In [ ]:
# Cell 12
# Save checkpoints + logits 

output_dir = CFG["output_dir"]

# Save per-seed checkpoints (needed for MCD in SA-2)
for seed in CFG["seeds"]:
    ckpt_path = os.path.join(output_dir, f"convnext_tiny_sea_animals_seed_{seed}.pth")
    torch.save({
        "model_state_dict": all_results[seed]["best_state"],
        "feature_dim": all_results[seed]["feature_dim"],
        "seed": seed,
        "model_name": CFG["model_name"],
        "num_classes": CFG["num_classes"],
    }, ckpt_path)
    print(f"Saved checkpoint: {ckpt_path} ({os.path.getsize(ckpt_path) / 1e6:.1f} MB)")

# Save all val/test logits
logits_path = os.path.join(output_dir, "sea_animals_logits.pth")
torch.save({
    "val_logits": {seed: saved_outputs[seed]["val_logits"] for seed in CFG["seeds"]},
    "val_labels": saved_outputs[CFG["seeds"][0]]["val_labels"],
    "test_logits": {seed: saved_outputs[seed]["test_logits"] for seed in CFG["seeds"]},
    "test_labels": saved_outputs[CFG["seeds"][0]]["test_labels"],
    "train_indices": train_indices,
    "val_indices": val_indices,
    "test_indices": test_indices,
    "all_image_paths": all_image_paths,
    "all_labels": all_labels,
    "label_names": class_names,
    "config": CFG,
}, logits_path)
print(f"\nSaved logits: {logits_path} ({os.path.getsize(logits_path) / 1e6:.1f} MB)")

# Save metrics
df_metrics.to_csv(os.path.join(output_dir, "sea_animals_seed_metrics.csv"), index=False)
print("Saved metrics CSV")

# List outputs
print(f"\nAll outputs in {output_dir}:")
for f in sorted(os.listdir(output_dir)):
    fpath = os.path.join(output_dir, f)
    if os.path.isfile(fpath):
        print(f"  {f} ({os.path.getsize(fpath) / 1e6:.1f} MB)")

In [ ]:
# Cell 13
print("Notebook SA-1 complete.")